## Import Libraries

In [46]:
import os
import time
import re
import pandas as pd
import requests
from tqdm import tqdm
from bs4 import BeautifulSoup
from dotenv import load_dotenv

## Authentication / Configuration

In [47]:
load_dotenv()  # Load .env file (contains API keys)

CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")

API_HEADERS = {
    "X-Naver-Client-Id": CLIENT_ID,
    "X-Naver-Client-Secret": CLIENT_SECRET
}

# Keywords for September 2025 news search
SEARCH_KEYWORDS = ["경제", "물가", "인플레이션", "금리", "부동산", "경기침체"]

# Output directory & file
SAVE_DIR = "data"
SAVE_PATH = os.path.join(SAVE_DIR, "economic_news_sep2025.csv")
os.makedirs(SAVE_DIR, exist_ok=True)

## Query Builder

In [48]:
def build_search_query(keyword: str) -> str:
    """Attach month and year context to base keyword."""
    return f"2025년 9월 {keyword}"

## Fetching News via Naver API

In [49]:
def fetch_naver_news(query: str, display: int = 100, start: int = 1, sort: str = "date"):
    """Fetch news results from Naver OpenAPI."""
    base_url = "https://openapi.naver.com/v1/search/news.json"
    params = {"query": query, "display": display, "start": start, "sort": sort}

    res = requests.get(base_url, headers=API_HEADERS, params=params)
    if res.status_code == 200:
        return res.json().get("items", [])
    else:
        print(f"[API Error {res.status_code}] {res.text}")
        return []

## Extract Full Article Text from Link

In [50]:
def extract_article_body(article_url: str) -> str | None:
    """Crawl and extract main article text using domain-specific selectors."""
    try:
        res = requests.get(article_url, headers={"User-Agent": "Mozilla/5.0"})
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")

        # Domain-based content selector map
        domain_selectors = {
            "naver.com": ["#dic_area", ".newsct_article"],
            "ajunews.com": [".article_body", "#articleBody"],
            "asiatoday.co.kr": [".article_body", "#textBody"],
            "g-enews.com": ["#articleBody", ".content_box"],
            "metroseoul.co.kr": [".view_con", ".art_txt"],
            "hansbiz.co.kr": [".article-body"],
            "thepublic.kr": [".article-body", "#article-view-content-div"],
            "lawissue.co.kr": ["#article-view-content-div", ".article-body"],
            "daily.hankooki.com": [".article-body", "#article-view-content-div"],
        }

        content_tag = None
        for domain, selectors in domain_selectors.items():
            if domain in article_url:
                for sel in selectors:
                    content_tag = soup.select_one(sel)
                    if content_tag:
                        break
                break

        if not content_tag:
            content_tag = soup.find("article")

        if content_tag:
            raw_text = content_tag.get_text(separator=" ", strip=True)
            clean_text = re.sub(r"\[.*?\]|\(.*?\)|사진.*?기자", "", raw_text)
            return clean_text

        return None

    except Exception as e:
        print(f"[Error fetching {article_url}]: {e}")
        return None

## Main Crawling Routine

In [51]:
def collect_economic_news():
    """Iterate over keywords and collect article data."""
    collected_data = []

    for kw in tqdm(SEARCH_KEYWORDS, desc="Keywords (September 2025)"):
        query = build_search_query(kw)

        for start in range(1, 1000, 100):  # fetch up to 100 per keyword (test mode)
            items = fetch_naver_news(query, display=100, start=start)
            if not items:
                break

            for it in items:
                article_body = extract_article_body(it["link"])
                collected_data.append({
                    "keyword": kw,
                    "title": it["title"].replace("<b>", "").replace("</b>", ""),
                    "summary": it["description"].replace("<b>", "").replace("</b>", ""),
                    "link": it["link"],
                    "originallink": it.get("originallink", ""),
                    "pubDate": it.get("pubDate", None),
                    "content": article_body
                })

            time.sleep(0.5)  # avoid rate limit

    return collected_data

## Save Results to CSV

In [52]:
def save_to_csv(data: list[dict], save_path: str):
    """Convert results to DataFrame and save as CSV."""
    df = pd.DataFrame(data)
    df.drop_duplicates(subset=["link"], inplace=True)
    df.reset_index(drop=True, inplace=True)

    if "pubDate" in df.columns:
        df["pubDate"] = pd.to_datetime(df["pubDate"], errors="coerce")

    df.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df)} rows to {save_path}")
    return df

## Execution

In [53]:
if __name__ == "__main__":
    results = collect_economic_news()
    final_df = save_to_csv(results, SAVE_PATH)
    display(final_df.head())

Keywords (September 2025):   0%|                                                                 | 0/6 [00:00<?, ?it/s]

[Error fetching http://www.ifs.or.kr/bbs/board.php?bo_table=globalnewsfocus&wr_id=124]: HTTPSConnectionPool(host='www.ifs.or.kr', port=443): Max retries exceeded with url: /bbs/board.php?bo_table=globalnewsfocus&wr_id=124 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1007)')))
[Error fetching http://www.ifs.or.kr/bbs/board.php?bo_table=globalnewsfocus&wr_id=120]: HTTPSConnectionPool(host='www.ifs.or.kr', port=443): Max retries exceeded with url: /bbs/board.php?bo_table=globalnewsfocus&wr_id=120 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1007)')))
[Error fetching https://www.artinsight.co.kr/news/view.php?no=77790]: HTTPSConnectionPool(host='www.artinsight.co.kr', port=443): Max retries exceeded with url: /news/view.php?no=77790 (Caused by SSLError(SSLCertVerification

Keywords (September 2025):  17%|█████████▎                                              | 1/6 [04:52<24:22, 292.47s/it]

[Error fetching http://www.ifs.or.kr/bbs/board.php?bo_table=globalnewsfocus&wr_id=120]: HTTPSConnectionPool(host='www.ifs.or.kr', port=443): Max retries exceeded with url: /bbs/board.php?bo_table=globalnewsfocus&wr_id=120 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1007)')))
[Error fetching http://www.ifs.or.kr/bbs/board.php?bo_table=research&wr_id=11151]: HTTPSConnectionPool(host='www.ifs.or.kr', port=443): Max retries exceeded with url: /bbs/board.php?bo_table=research&wr_id=11151 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1007)')))
[Error fetching http://www.ifs.or.kr/bbs/board.php?bo_table=globalnewsfocus&wr_id=112]: HTTPSConnectionPool(host='www.ifs.or.kr', port=443): Max retries exceeded with url: /bbs/board.php?bo_table=globalnewsfocus&wr_id=112 (Caused by S

Keywords (September 2025):  33%|██████████████████▋                                     | 2/6 [09:19<18:28, 277.23s/it]

...

Keywords (September 2025):  83%|██████████████████████████████████████████████▋         | 5/6 [39:00<10:13, 613.51s/it]

[Error fetching https://www.bntnews.co.kr/article/view/bnt202509260049]: 502 Server Error: Bad Gateway for url: https://www.bntnews.co.kr/article/view/bnt202509260049
[Error fetching http://www.ifs.or.kr/bbs/board.php?bo_table=WallWatch&wr_id=120]: HTTPSConnectionPool(host='www.ifs.or.kr', port=443): Max retries exceeded with url: /bbs/board.php?bo_table=WallWatch&wr_id=120 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1007)')))
[Error fetching https://www.bntnews.co.kr/article/view/bnt202509190049]: 502 Server Error: Bad Gateway for url: https://www.bntnews.co.kr/article/view/bnt202509190049
[Error fetching https://vop.co.kr/A00001679461.html]: HTTPSConnectionPool(host='vop.co.kr', port=443): Max retries exceeded with url: /A00001679461.html (Caused by SSLError(SSLError(1, '[SSL: DH_KEY_TOO_SMALL] dh key too small (_ssl.c:1007)')))
[Error fetching https://www.bntnews.co.kr/art

Keywords (September 2025): 100%|████████████████████████████████████████████████████████| 6/6 [43:26<00:00, 434.37s/it]


Saved 4610 rows to data\economic_news_sep2025.csv


,keyword,title,summary,link,originallink,pubDate,content
0,경제,이번주 국내 주요 금융일정(10.13~10.17),"한국은행, 2025년 9월 중 금융시장 동향(12시) 한국은행, 2/4분기 자금순환...",https://www.newspim.com/news/view/20251010000803,https://www.newspim.com/news/view/20251010000803,2025-10-11 17:00:00+09:00,이름부터 제다이 ② 드론과 AI가 이끄는 차세대 방산 ETF
1,경제,"완주군, 교육발전특구 연계로 '정주형 농촌유학' 선도",지역경제 활성화로 이어지는 완주형 지속가능 농촌모델의 핵심 축으로 평가받고 있다. ...,https://www.goodnews1.com/news/articleView.htm...,https://www.goodnews1.com/news/articleView.htm...,2025-10-11 16:16:00+09:00,"지역본부뉴스 완주군, 교육발전특구 연계로 ‘정주형 농촌유학’ 선도 숙소‧생활공간 리..."
2,경제,"비트코인 vs 이더리움, 2025년 승자는? 상승률 앞선 ETH에도 중심축은 B...",이더리움은 2025년 현재까지 연초 대비 약 30% 상승하며 비트코인(+25%)을 ...,https://www.tokenpost.kr/news/cryptocurrency/2...,https://www.tokenpost.kr/news/cryptocurrency/2...,2025-10-11 16:06:00+09:00,링크가 복사되었습니다. 공유 페이스북 엑스 링크드인 텔레그램 글자크기 가 작게 가 ...
3,경제,발전공기업 태양광·풍력 보급계획 가속화 필요…2040년 탈석탄 대비해...,11일 한국전력 경영연구원이 최근 발간한 '2024년 글로벌 재생에너지 발전용량 및...,https://daily.hankooki.com/news/articleView.ht...,https://daily.hankooki.com/news/articleView.ht...,2025-10-11 15:00:00+09:00,"한전 경영연구원, 2024년 글로벌 보급 역대 최대 규모 증가 글로벌 태양광·풍력 ..."
4,경제,9월 글로벌 선박 수주량 44% 급감…K-조선 39% '세계 2위',/연합뉴스 | 한스경제=이성노 기자 | 지난달 글로벌 선박 수주량이 전년 동기 대비...,http://www.hansbiz.co.kr/news/articleView.html...,http://www.hansbiz.co.kr/news/articleView.html...,2025-10-11 14:12:00+09:00,내용요약 글로벌 수주량 350만CGT…韓 135만CGT 지난달 글로벌 선박 수주량...
